# AWS Translate – translate Bangla Prothom Alo articles to English

In [2]:
# Import required libraries and set up our environment
import pandas as pd
from pprint import PrettyPrinter

import boto3
import time

print("📚 Setting up the environment...")
pp = PrettyPrinter(indent=2)
translate = boto3.client("translate")
print("✅ Environment setup complete!")
print(f"🌍 Using AWS region: {translate.meta.region_name}")
print("✅ AWS Translate client ready")

📚 Setting up the environment...
✅ Environment setup complete!
🌍 Using AWS region: eu-west-1
✅ AWS Translate client ready


In [3]:
# Load the dataset

print("📄 Loading articles.csv...")

df = pd.read_csv("articles.csv")

print("✅ Dataset loaded")
print("Columns:", list(df.columns))
print("Language counts:")
print(df["language"].value_counts())


📄 Loading articles.csv...
✅ Dataset loaded
Columns: ['source', 'language', 'url', 'article_text', 'text_orig_ln', 'translated_text']
Language counts:
language
en    18
bn    12
Name: count, dtype: int64


#### Support function (AWS Translate call)

In [4]:
def translate_to_en(text, source_lang):
    """
    Translate text to English using Amazon Translate.
    Safely handles empty or NaN values.
    """
    if text is None or (isinstance(text, float) and pd.isna(text)) or str(text).strip() == "":
        return None

    response = translate.translate_text(
        Text=str(text)[:5000],            # AWS limit
        SourceLanguageCode=source_lang,
        TargetLanguageCode="en"
    )

    return response["TranslatedText"]


#### Translation UDF

In [5]:
def translate_non_english_rows(
    df,
    text_col="article_text",
    lang_col="language",
    sleep_s=0.1
):
    df = df.copy()

    # Ensure bookkeeping columns exist
    if "text_orig_ln" not in df.columns:
        df["text_orig_ln"] = pd.NA
    if "translated_text" not in df.columns:
        df["translated_text"] = pd.NA
    if "translate_error" not in df.columns:
        df["translate_error"] = pd.NA

    # Mask for non-English rows
    mask = (
        df[lang_col].notna()
        & (df[lang_col].str.lower() != "en")
        & df[text_col].notna()
    )

    # Freeze original non-English text
    df.loc[mask, "text_orig_ln"] = df.loc[mask, text_col]

    print("🔁 Translating non-English articles...")

    for idx, row in df.loc[mask, [text_col, lang_col]].iterrows():
        try:
            df.at[idx, "translated_text"] = translate_to_en(
                row[text_col],
                row[lang_col].lower()
            )
            print(f"✅ Translated row {idx}")

        except Exception as e:
            df.at[idx, "translate_error"] = str(e)
            print(f"❌ Translation failed at row {idx}: {str(e)}")

        if sleep_s:
            time.sleep(sleep_s)

    # English rows → copy text directly
    en_mask = df[lang_col].str.lower() == "en"
    df.loc[en_mask, "translated_text"] = df.loc[en_mask, text_col]

    return df


#### Run Translation

In [6]:
translated_df = translate_non_english_rows(
    df,
    text_col="article_text",
    lang_col="language",
    sleep_s=0.1
)



🔁 Translating non-English articles...
✅ Translated row 10
✅ Translated row 11
✅ Translated row 12
✅ Translated row 13
✅ Translated row 14
✅ Translated row 15
✅ Translated row 16
✅ Translated row 17
✅ Translated row 18
✅ Translated row 21
✅ Translated row 24
✅ Translated row 26


In [8]:
translated_df.to_csv("articles.csv", index=False)

print("💾 Translated dataset saved successfully")


💾 Translated dataset saved successfully


In [10]:
translated_df[["language", "article_text", "translated_text"]].head(30)


,language,article_text,translated_text
0,en,The interim government's revised election time...,The interim government's revised election time...
1,en,BNP has announced its candidates for 36 more c...,BNP has announced its candidates for 36 more c...
2,en,Discontent runs deep among BNP's partners as t...,Discontent runs deep among BNP's partners as t...
3,en,Even though a week has passed since the announ...,Even though a week has passed since the announ...
4,en,The next national election will be tough for t...,The next national election will be tough for t...
5,en,"An opinion piece titled ""BNP's notes of dissen...","An opinion piece titled ""BNP's notes of dissen..."
6,en,The BNP has survived Sheikh Hasina's 15-year r...,The BNP has survived Sheikh Hasina's 15-year r...
7,en,The former Prime Minister Khaleda Zia on Tuesd...,The former Prime Minister Khaleda Zia on Tuesd...
8,en,After having weathered a difficult 15 years in...,After having weathered a difficult 15 years in...
9,en,The BNP has alerted its election candidates th...,The BNP has alerted its election candidates th...
